# 09 — Generate main paper figures and tables

This notebook is the manuscript production layer. It reads finalized products
from Notebooks 01–08 and generates the proposed main-text figures, primary
tables, and selected supplementary diagnostics. It does **not** remeasure
waveforms or rerun array inversions.

Run Notebooks 01–08 sequentially before using these products for publication.


## 1. Imports, paths, style, and finalized inputs


In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
import pandas as pd
from matplotlib.colors import LogNorm
from obspy import Stream, Trace, UTCDateTime, read
from pyproj import CRS, Geod, Transformer
from scipy.stats import linregress

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR
MODULE_DIR = PROJECT_ROOT / "modules"
if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))

import figure1_utils
from figure1_utils import get_sensor_location, make_figure1, save_figure
from geometry_products import read_geometry_products
from plotting import plot_key_event_waveforms

DERIVED_DIR = PROJECT_ROOT / "outputs" / "derived"
PREPARATION_ROOT = PROJECT_ROOT / "outputs" / "candidate_airwave_preparation"
PLANAR_ROOT = PROJECT_ROOT / "outputs" / "array_analysis_planar_refined"
SPHERICAL_ROOT = PROJECT_ROOT / "outputs" / "array_analysis_circular"
AMPLITUDE_ROOT = PROJECT_ROOT / "outputs" / "event_amplitude_coupling"
RESPONSE_ROOT = PROJECT_ROOT / "outputs" / "response_correction" / "final_products"

PRODUCT_ROOT = PROJECT_ROOT / "outputs" / "paper_products"
MAIN_FIGURE_DIR = PRODUCT_ROOT / "main_figures"
SUPPLEMENT_FIGURE_DIR = PRODUCT_ROOT / "supplementary_figures"
TABLE_DIR = PRODUCT_ROOT / "tables"
for directory in (PRODUCT_ROOT, MAIN_FIGURE_DIR, SUPPLEMENT_FIGURE_DIR, TABLE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

ANALYSIS_CONFIG_FILE = DERIVED_DIR / "analysis_configuration.json"
GEOMETRY_FILE = DERIVED_DIR / "bchh_geometry.csv"
KEY_EVENTS_FILE = DERIVED_DIR / "key_events.csv"
CANDIDATE_FILE = PREPARATION_ROOT / "derived" / "candidate_airwave_event_catalogue.csv"
BASELINE_STREAM_FILE = DERIVED_DIR / "bchh_corrected_moving_median_baseline_removed.pkl"
PLANAR_FILE = PLANAR_ROOT / "planar_array_results.csv"
DECISION_FILE = PLANAR_ROOT / "event_filter_decisions.csv"
AMPLITUDE_FILE = AMPLITUDE_ROOT / "event_acoustic_seismic_amplitudes.csv"
SPHERICAL_FILE = SPHERICAL_ROOT / "fixed_slc40_spherical_results.csv"
RANGE_PROFILE_FILE = SPHERICAL_ROOT / "range_profile_results.csv"

# Synchronized public-video audio used only as a normalized visual timing aid.
YOUTUBE_AUDIO_FILE = Path(
    "/Users/thompsong/Library/CloudStorage/Box-Box/thompsong/"
    "3_Project_Documents/NASAprojects/201602_Rocket_Seismology/02_KSC/"
    "spaceX_explosion_movie/syncedsound2.mp3"
)
YOUTUBE_AUDIO_STARTTIME = UTCDateTime("2016-09-01T13:07:10.000")
YOUTUBE_AUDIO_BAND_HZ = (1000.0, 2000.0)
YOUTUBE_AUDIO_REDUCED_TIME_SHIFT_S = 3.755

required = [
    ANALYSIS_CONFIG_FILE, GEOMETRY_FILE, KEY_EVENTS_FILE, CANDIDATE_FILE,
    BASELINE_STREAM_FILE, PLANAR_FILE, DECISION_FILE, AMPLITUDE_FILE,
    SPHERICAL_FILE, RANGE_PROFILE_FILE,
]
missing = [path for path in required if not path.exists()]
if missing:
    raise FileNotFoundError("Missing finalized inputs:\n" + "\n".join(map(str, missing)))

analysis_config = json.loads(ANALYSIS_CONFIG_FILE.read_text())
geometry = pd.read_csv(GEOMETRY_FILE)
key_events = pd.read_csv(KEY_EVENTS_FILE)
candidates = pd.read_csv(CANDIDATE_FILE).rename(
    columns={"candidate_event_number": "event_number"}
)
decisions = pd.read_csv(DECISION_FILE)
planar = pd.read_csv(PLANAR_FILE)
measurements = pd.read_csv(AMPLITUDE_FILE)
spherical = pd.read_csv(SPHERICAL_FILE)
range_profiles = pd.read_csv(RANGE_PROFILE_FILE)
st_baseline = read(str(BASELINE_STREAM_FILE), format="PICKLE")

for frame in (candidates, decisions, planar, measurements, spherical, range_profiles):
    frame["event_number"] = frame["event_number"].astype(int)

if "event_time" in measurements.columns:
    measurements["event_time"] = pd.to_datetime(
        measurements["event_time"], utc=True
    )
else:
    measurements["event_time"] = pd.to_datetime(
        measurements["reduced_event_epoch_s"], unit="s", utc=True
    )
measurements["elapsed_time_min"] = (
    measurements["reduced_event_epoch_s"]
    - measurements["reduced_event_epoch_s"].min()
) / 60.0

PRINCIPAL_EVENT_NUMBER = 13
KEY_EVENT_NUMBERS = {
    "second_stage": 2,
    "principal_explosion": 13,
    "capsule_impact": 22,
    "capsule_explosion": 23,
}
KEY_EVENT_DISPLAY = {
    "second_stage": "Initial second-stage failure",
    "principal_explosion": "Principal explosion",
    "capsule_pulse_1": "Capsule pulse 1",
    "capsule_pulse_2": "Capsule pulse 2",
}

TIER_STYLE = {
    "core": dict(marker="o", s=30, alpha=0.85, color="tab:blue", label="Core planar set"),
    "additional_analyzed": dict(marker="^", s=28, alpha=0.75, color="tab:green", label="Additional analyzed"),
    "other_candidate": dict(marker=".", s=18, alpha=0.35, color="0.55", label="Other candidate"),
}

plt.rcParams.update({
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
    "legend.fontsize": 8,
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

def save_main(fig, stem):
    for extension in ("png", "pdf"):
        fig.savefig(MAIN_FIGURE_DIR / f"{stem}.{extension}", bbox_inches="tight")

def save_supplement(fig, stem):
    for extension in ("png", "pdf"):
        fig.savefig(SUPPLEMENT_FIGURE_DIR / f"{stem}.{extension}", bbox_inches="tight")

print(f"Candidates: {len(candidates)}")
print(f"Planar results: {len(planar)}")
print(f"Core events: {int(decisions['passes_core_filter'].sum())}")
print(f"Amplitude measurements: {len(measurements)}")


## 2. Figure 1 — SLC-40 and BCHH deployment geometry


In [ ]:
inventory_event, channels_df, stations_df, locations_df = read_geometry_products(DERIVED_DIR)

channel_to_sensor = {
    "DHZ": "Seismometer", "DHN": "Seismometer", "DHE": "Seismometer",
    "HHZ": "Seismometer", "HHN": "Seismometer", "HHE": "Seismometer",
    "DD1": "HD1", "DD2": "HD2", "DD3": "HD3",
    "HD1": "HD1", "HD2": "HD2", "HD3": "HD3",
}
channel_to_label = {key: ("BCHH" if value == "Seismometer" else value)
                    for key, value in channel_to_sensor.items()}

channels_for_map = channels_df.copy()
codes = channels_for_map["channel"].astype(str).str.upper()
channels_for_map["sensor"] = codes.map(channel_to_sensor)
channels_for_map["label"] = codes.map(channel_to_label)
channels_for_map["lat"] = channels_for_map["latitude"].astype(float)
channels_for_map["lon"] = channels_for_map["longitude"].astype(float)

wgs84 = CRS.from_epsg(4326)
utm17n = CRS.from_epsg(32617)
ll_to_utm = Transformer.from_crs(wgs84, utm17n, always_xy=True)
utm_to_ll = Transformer.from_crs(utm17n, wgs84, always_xy=True)
east, north = ll_to_utm.transform(channels_for_map["lon"], channels_for_map["lat"])
channels_for_map["easting"] = east
channels_for_map["northing"] = north

mapped = channels_for_map.dropna(subset=["sensor"])
seismometer = mapped.loc[mapped["sensor"].eq("Seismometer")].iloc[[0]]
infrasound = (
    mapped.loc[mapped["sensor"].isin(["HD1", "HD2", "HD3"])]
    .sort_values("sensor").drop_duplicates("sensor")
)
bchh_sensors = pd.concat([seismometer, infrasound], ignore_index=True)

def kml_location(kml_id):
    matches = locations_df.loc[locations_df["kml_id"].astype(str).str.upper().eq(kml_id.upper())]
    if len(matches) != 1:
        raise ValueError(f"Expected one {kml_id}; found {len(matches)}")
    row = matches.iloc[0]
    lat_key = "lat" if "lat" in row.index else "latitude"
    lon_key = "lon" if "lon" in row.index else "longitude"
    return {**row.to_dict(), "name": str(row.get("name", kml_id)),
            "lat": float(row[lat_key]), "lon": float(row[lon_key])}

slc40 = kml_location("SLC40")
slc41 = kml_location("SLC41")
bchh = get_sensor_location(bchh_sensors, sensor_name="Seismometer", display_name="BCHH")
make_kwargs = dict(
    slc40=slc40, slc41=slc41, bchh=bchh, bchh_sensors=bchh_sensors,
    geod=Geod(ellps="WGS84"), ll_to_utm=ll_to_utm, utm_to_ll=utm_to_ll,
)
original_add_basemap = figure1_utils.add_basemap
if os.environ.get("OFFLINE_BASEMAP", "0") == "1":
    figure1_utils.add_basemap = lambda *args, **kwargs: None
try:
    fig, axes = make_figure1(**make_kwargs)
except Exception as exc:
    # Preserve a reproducible geometry figure on offline systems. The user can
    # rerun later with network access to restore the contextual map tiles.
    plt.close("all")
    print(f"Basemap unavailable ({exc}); drawing geometry without map tiles.")
    figure1_utils.add_basemap = lambda *args, **kwargs: None
    fig, axes = make_figure1(**make_kwargs)
finally:
    figure1_utils.add_basemap = original_add_basemap
save_main(fig, "fig01_slc40_bchh_geometry")
plt.show()


> **Figure 1.** Location and geometry of the BCHH seismo-acoustic array. **(A)** Regional setting at Kennedy Space Center, showing Space Launch Complex 40 (SLC-40; yellow triangle), the BCHH array near the Astronaut Beach House (black circle), and neighboring SLC-41. BCHH was approximately 1.42 km north-northeast of the Falcon 9 vehicle, along an azimuth of approximately 19° from the launch pad; the reciprocal back azimuth from the array to SLC-40 was approximately 199°. **(B)** Differential-GPS-surveyed BCHH array geometry. Three infraBSU pressure sensors (HD1–HD3; open circles) formed an approximately triangular array surrounding the three-component Trillium Compact broadband seismometer (black square), with an aperture of approximately 30 m. Geographic coordinates are shown on the lower and left axes, with UTM Zone 17N coordinates on the upper and right axes. Basemap data: OpenStreetMap contributors and CARTO.

In [ ]:
# Supplementary figure: launch pads, BCHH, and video-camera geometry.

from matplotlib.lines import Line2D
from matplotlib.ticker import FormatStrFormatter


CAMERA_LOCATIONS_FILE = (
    DERIVED_DIR / "launchpad_camera_locations.csv"
)

CAMERA_DEFINITIONS = {
    "YouTube Video": {
        "camera_id": "firey",
        "display_name": "YouTube",
        "used_audio": True,
    },
    "UCS-3": {
        "camera_id": "ucs3",
        "display_name": "UCS-3",
        "used_audio": False,
    },
    "NE Tower": {
        "camera_id": "ne_tower",
        "display_name": "NE Tower",
        "used_audio": False,
    },
    "Pad NW": {
        "camera_id": "pad_nw",
        "display_name": "Pad NW",
        "used_audio": False,
    },
    "Pad West": {
        "camera_id": "pad_west",
        "display_name": "Pad West",
        "used_audio": False,
    },
}

LAUNCHPAD_DEFINITIONS = {
    "SLC40": "SLC-40",
    "SLC41": "SLC-41",
}

NEAR_PAD_CAMERAS = {
    "NE Tower",
    "Pad NW",
    "Pad West",
}


if not CAMERA_LOCATIONS_FILE.exists():
    raise FileNotFoundError(
        f"Camera-location table not found: "
        f"{CAMERA_LOCATIONS_FILE}"
    )

location_table = pd.read_csv(CAMERA_LOCATIONS_FILE)

required_location_columns = {
    "kml_id",
    "lat",
    "lon",
}

missing_location_columns = (
    required_location_columns - set(location_table.columns)
)

if missing_location_columns:
    raise KeyError(
        "Camera-location table lacks required columns: "
        f"{sorted(missing_location_columns)}"
    )


def select_named_locations(dataframe, names):
    """Select one KML-derived location for every requested name."""
    selected_rows = []

    normalized_ids = (
        dataframe["kml_id"]
        .astype(str)
        .str.strip()
        .str.casefold()
    )

    for requested_name in names:
        matches = dataframe.loc[
            normalized_ids.eq(
                requested_name.strip().casefold()
            )
        ]

        if len(matches) != 1:
            raise ValueError(
                f"Expected one KML location named "
                f"{requested_name!r}; found {len(matches)}."
            )

        selected_rows.append(matches.iloc[0].copy())

    return pd.DataFrame(selected_rows).reset_index(drop=True)


camera_points = select_named_locations(
    location_table,
    CAMERA_DEFINITIONS,
)

camera_points["camera_id"] = camera_points[
    "kml_id"
].map(
    lambda value: CAMERA_DEFINITIONS[value]["camera_id"]
)

camera_points["display_name"] = camera_points[
    "kml_id"
].map(
    lambda value: CAMERA_DEFINITIONS[value][
        "display_name"
    ]
)

camera_points["used_audio"] = camera_points[
    "kml_id"
].map(
    lambda value: CAMERA_DEFINITIONS[value]["used_audio"]
)

launchpad_points = select_named_locations(
    location_table,
    LAUNCHPAD_DEFINITIONS,
)

launchpad_points["display_name"] = launchpad_points[
    "kml_id"
].map(LAUNCHPAD_DEFINITIONS)


# Use the surveyed BCHH seismometer position already prepared
# for Figure 1.
station_points = pd.DataFrame([{
    "station_id": "BCHH",
    "display_name": "BCHH",
    "lat": float(bchh["lat"]),
    "lon": float(bchh["lon"]),
}])


# Calculate source-to-receiver geometry from SLC-40.
geod = Geod(ellps="WGS84")

camera_azimuths = []
camera_distances = []

for camera in camera_points.itertuples(index=False):
    azimuth_deg, _, distance_m = geod.inv(
        float(slc40["lon"]),
        float(slc40["lat"]),
        float(camera.lon),
        float(camera.lat),
    )

    camera_azimuths.append(
        azimuth_deg % 360.0
    )
    camera_distances.append(distance_m)

camera_points["distance_from_slc40_m"] = (
    camera_distances
)
camera_points["azimuth_from_slc40_deg"] = (
    camera_azimuths
)


station_azimuths = []
station_distances = []

for station in station_points.itertuples(index=False):
    azimuth_deg, _, distance_m = geod.inv(
        float(slc40["lon"]),
        float(slc40["lat"]),
        float(station.lon),
        float(station.lat),
    )

    station_azimuths.append(
        azimuth_deg % 360.0
    )
    station_distances.append(distance_m)

station_points["distance_from_slc40_m"] = (
    station_distances
)
station_points["azimuth_from_slc40_deg"] = (
    station_azimuths
)


# Export the receiver geometry used by the figure.
camera_geometry_export = camera_points.loc[
    :,
    [
        "camera_id",
        "display_name",
        "lat",
        "lon",
        "altitude_m",
        "distance_from_slc40_m",
        "azimuth_from_slc40_deg",
        "used_audio",
    ],
].copy()

camera_geometry_export[
    "distance_from_slc40_m"
] = camera_geometry_export[
    "distance_from_slc40_m"
].round(1)

camera_geometry_export[
    "azimuth_from_slc40_deg"
] = camera_geometry_export[
    "azimuth_from_slc40_deg"
].round(1)

camera_geometry_export.to_csv(
    TABLE_DIR / "tableS_camera_geometry.csv",
    index=False,
)

display(camera_geometry_export)


def padded_limits(longitudes, latitudes, *,
                  longitude_fraction=0.08,
                  latitude_fraction=0.10,
                  minimum_longitude_padding=0.0003,
                  minimum_latitude_padding=0.0003):
    """Return padded longitude and latitude plotting limits."""
    longitudes = np.asarray(longitudes, dtype=float)
    latitudes = np.asarray(latitudes, dtype=float)

    longitude_span = np.ptp(longitudes)
    latitude_span = np.ptp(latitudes)

    longitude_padding = max(
        minimum_longitude_padding,
        longitude_span * longitude_fraction,
    )
    latitude_padding = max(
        minimum_latitude_padding,
        latitude_span * latitude_fraction,
    )

    return (
        (
            longitudes.min() - longitude_padding,
            longitudes.max() + longitude_padding,
        ),
        (
            latitudes.min() - latitude_padding,
            latitudes.max() + latitude_padding,
        ),
    )


def add_map_label(ax, longitude, latitude, text, *,
                  offset=(5, 5),
                  horizontal_alignment="left"):
    """Add a compact publication-style map label."""
    ax.annotate(
        text,
        xy=(longitude, latitude),
        xytext=offset,
        textcoords="offset points",
        ha=horizontal_alignment,
        va="bottom",
        fontsize=8,
        bbox={
            "facecolor": "white",
            "edgecolor": "none",
            "alpha": 0.82,
            "pad": 1.0,
        },
        zorder=50,
    )


def configure_geographic_axis(
    ax,
    longitude_limits,
    latitude_limits,
    *,
    panel_label,
    add_tiles,
):
    """Configure one longitude-latitude map panel."""
    ax.set_xlim(longitude_limits)
    ax.set_ylim(latitude_limits)

    ax.set_xlabel("Longitude (°W)")
    ax.set_ylabel("Latitude (°N)")

    ax.xaxis.set_major_formatter(
        FormatStrFormatter("%.3f")
    )
    ax.yaxis.set_major_formatter(
        FormatStrFormatter("%.3f")
    )

    figure1_utils.set_equal_ground_aspect(ax)

    if add_tiles:
        figure1_utils.add_basemap(
            ax,
            provider="voyager",
            alpha=0.78,
            zoom="auto",
        )

        # Restore exact limits after adding tiles.
        ax.set_xlim(longitude_limits)
        ax.set_ylim(latitude_limits)

    ax.set_title(
        panel_label,
        loc="left",
        fontsize=13,
        fontweight="bold",
    )


def draw_receiver_map(add_tiles=True):
    """Draw regional and near-pad receiver geometry."""
    fig, axes = plt.subplots(
        1,
        2,
        figsize=(13.0, 6.0),
        gridspec_kw={
            "width_ratios": [1.25, 1.0],
        },
    )

    fig.subplots_adjust(
        left=0.065,
        right=0.985,
        bottom=0.12,
        top=0.90,
        wspace=0.23,
    )

    # ---------------------------------------------------------
    # Panel A: regional source-receiver geometry
    # ---------------------------------------------------------
    regional_camera_points = camera_points.copy()

    regional_longitudes = np.concatenate([
        regional_camera_points["lon"].to_numpy(),
        launchpad_points["lon"].to_numpy(),
        station_points["lon"].to_numpy(),
    ])

    regional_latitudes = np.concatenate([
        regional_camera_points["lat"].to_numpy(),
        launchpad_points["lat"].to_numpy(),
        station_points["lat"].to_numpy(),
    ])

    regional_lonlim, regional_latlim = padded_limits(
        regional_longitudes,
        regional_latitudes,
        longitude_fraction=0.07,
        latitude_fraction=0.09,
        minimum_longitude_padding=0.002,
        minimum_latitude_padding=0.002,
    )

    ax = axes[0]

    configure_geographic_axis(
        ax,
        regional_lonlim,
        regional_latlim,
        panel_label="A",
        add_tiles=add_tiles,
    )

    # Source-to-receiver paths.
    for receiver in camera_points.itertuples(index=False):
        ax.plot(
            [slc40["lon"], receiver.lon],
            [slc40["lat"], receiver.lat],
            color="0.45",
            lw=0.75,
            alpha=0.65,
            zorder=8,
        )

    for receiver in station_points.itertuples(index=False):
        ax.plot(
            [slc40["lon"], receiver.lon],
            [slc40["lat"], receiver.lat],
            color="0.15",
            lw=1.2,
            alpha=0.8,
            zorder=9,
        )

    # Launch complexes.
    for pad in launchpad_points.itertuples(index=False):
        is_source = pad.kml_id == "SLC40"

        ax.scatter(
            pad.lon,
            pad.lat,
            marker="^",
            s=165 if is_source else 120,
            facecolor="gold" if is_source else "white",
            edgecolor="black",
            linewidth=1.0,
            zorder=30,
        )

        add_map_label(
            ax,
            pad.lon,
            pad.lat,
            pad.display_name,
            offset=(5, 4),
        )

    # BCHH.
    for station in station_points.itertuples(index=False):
        ax.scatter(
            station.lon,
            station.lat,
            marker="D",
            s=90,
            facecolor="black",
            edgecolor="black",
            zorder=32,
        )

        add_map_label(
            ax,
            station.lon,
            station.lat,
            station.display_name,
            offset=(6, 4),
        )

    # Camera locations.
    for camera in camera_points.itertuples(index=False):
        ax.scatter(
            camera.lon,
            camera.lat,
            marker="s",
            s=85,
            facecolor=(
                "tab:red"
                if camera.used_audio
                else "white"
            ),
            edgecolor="tab:red",
            linewidth=1.2,
            zorder=31,
        )

        # The three pad cameras are labeled together in the
        # regional panel and individually in Panel B.
        if camera.kml_id not in NEAR_PAD_CAMERAS:
            add_map_label(
                ax,
                camera.lon,
                camera.lat,
                camera.display_name,
                offset=(6, 4),
            )

    near_pad_lon = camera_points.loc[
        camera_points["kml_id"].isin(
            NEAR_PAD_CAMERAS
        ),
        "lon",
    ].mean()

    near_pad_lat = camera_points.loc[
        camera_points["kml_id"].isin(
            NEAR_PAD_CAMERAS
        ),
        "lat",
    ].mean()

    add_map_label(
        ax,
        near_pad_lon,
        near_pad_lat,
        "",#"near-pad cameras",
        offset=(7, -14),
    )

    figure1_utils.add_scalebar_lonlat(
        ax=ax,
        lon=regional_lonlim[0] + 0.004,
        lat=regional_latlim[0] + 0.013,
        length_m=1000,
        label_text="1 km",
        geod=geod,
    )

    figure1_utils.add_north_arrow_axes(
        ax,
        xy=(0.93, 0.82),
        length=0.09,
    )

    # ---------------------------------------------------------
    # Panel B: near-pad camera geometry
    # ---------------------------------------------------------
    local_cameras = camera_points.loc[
        camera_points["kml_id"].isin(
            NEAR_PAD_CAMERAS
        )
    ].copy()

    slc40_point = launchpad_points.loc[
        launchpad_points["kml_id"].eq("SLC40")
    ]

    local_longitudes = np.concatenate([
        local_cameras["lon"].to_numpy(),
        slc40_point["lon"].to_numpy(),
    ])

    local_latitudes = np.concatenate([
        local_cameras["lat"].to_numpy(),
        slc40_point["lat"].to_numpy(),
    ])

    local_lonlim, local_latlim = padded_limits(
        local_longitudes,
        local_latitudes,
        longitude_fraction=0.20,
        latitude_fraction=0.30,
        minimum_longitude_padding=0.00025,
        minimum_latitude_padding=0.00020,
    )

    ax = axes[1]

    configure_geographic_axis(
        ax,
        local_lonlim,
        local_latlim,
        panel_label="B",
        add_tiles=add_tiles,
    )

    ax.xaxis.set_major_formatter(
        FormatStrFormatter("%.4f")
    )
    ax.yaxis.set_major_formatter(
        FormatStrFormatter("%.4f")
    )

    for camera in local_cameras.itertuples(index=False):
        ax.plot(
            [slc40["lon"], camera.lon],
            [slc40["lat"], camera.lat],
            color="0.45",
            lw=0.9,
            alpha=0.75,
            zorder=8,
        )

    ax.scatter(
        slc40["lon"],
        slc40["lat"],
        marker="^",
        s=185,
        facecolor="gold",
        edgecolor="black",
        linewidth=1.0,
        zorder=30,
    )

    add_map_label(
        ax,
        slc40["lon"],
        slc40["lat"],
        "SLC-40",
        offset=(5, -15),
    )

    local_label_offsets = {
        "NE Tower": (6, 5),
        "Pad NW": (-5, 6),
        "Pad West": (-5, -16),
    }

    local_label_alignment = {
        "NE Tower": "left",
        "Pad NW": "right",
        "Pad West": "right",
    }

    for camera in local_cameras.itertuples(index=False):
        ax.scatter(
            camera.lon,
            camera.lat,
            marker="s",
            s=90,
            facecolor="white",
            edgecolor="tab:red",
            linewidth=1.3,
            zorder=31,
        )

        add_map_label(
            ax,
            camera.lon,
            camera.lat,
            camera.display_name,
            offset=local_label_offsets[
                camera.kml_id
            ],
            horizontal_alignment=(
                local_label_alignment[
                    camera.kml_id
                ]
            ),
        )

    figure1_utils.add_scalebar_lonlat(
        ax=ax,
        lon=local_lonlim[0] + 0.00025,
        lat=local_latlim[0] + 0.00016,
        length_m=100,
        label_text="100 m",
        geod=geod,
    )

    figure1_utils.add_north_arrow_axes(
        ax,
        xy=(0.92, 0.81),
        length=0.10,
    )

    # One shared legend.
    legend_handles = [
        Line2D(
            [0], [0],
            marker="^",
            linestyle="none",
            markersize=9,
            markerfacecolor="gold",
            markeredgecolor="black",
            label="SLC-40",
        ),
        Line2D(
            [0], [0],
            marker="^",
            linestyle="none",
            markersize=8,
            markerfacecolor="white",
            markeredgecolor="black",
            label="Other launch complex",
        ),
        Line2D(
            [0], [0],
            marker="D",
            linestyle="none",
            markersize=7,
            markerfacecolor="black",
            markeredgecolor="black",
            label="BCHH seismo-acoustic array",
        ),
        Line2D(
            [0], [0],
            marker="s",
            linestyle="none",
            markersize=7,
            markerfacecolor="tab:red",
            markeredgecolor="tab:red",
            label="YouTube Camera",
        ),
        Line2D(
            [0], [0],
            marker="s",
            linestyle="none",
            markersize=7,
            markerfacecolor="white",
            markeredgecolor="tab:red",
            label="Other camera",
        ),
    ]

    axes[0].legend(
        handles=legend_handles,
        loc="upper left",
        frameon=True,
        fontsize=7.5,
    )

    return fig, axes


# Try the contextual basemap first, but retain a reproducible
# geometry-only fallback for offline execution.
try:
    fig, axes = draw_receiver_map(add_tiles=True)
except Exception as exc:
    plt.close("all")
    print(
        "Basemap unavailable; drawing receiver geometry "
        f"without map tiles. Reason: {exc}"
    )
    fig, axes = draw_receiver_map(add_tiles=False)

save_supplement(
    fig,
    "figS_camera_station_launchpad_geometry",
)

plt.show()

Figure Sx. Locations of the launch complex, BCHH seismo-acoustic array, and video cameras used to interpret the Falcon 9 accident sequence. A: regional geometry showing SLC-40, BCHH, the FIREY/YouTube camera, UCS-3, and neighboring launch complexes. B: enlarged view of SLC-40 and the nearby NE Tower, Pad NW, and Pad West cameras. Lines connect SLC-40 to the principal receivers and illustrate the distinct acoustic propagation paths. Camera and launch-pad coordinates were obtained from the project KML metadata; the BCHH position was determined by differential GPS survey. The FIREY camera supplied the YouTube audio used as an approximate timing aid. Because the video files lack absolute timestamps, camera audio was aligned empirically and was not used as an independently timed quantitative observation.


Suggested caption:

> **Figure Sx.** Locations of the launch complex, BCHH seismo-acoustic array, and video cameras used to interpret the Falcon 9 accident sequence. **A:** regional geometry showing SLC-40, SLC-41, BCHH, the YouTube camera, UCS-3, and the cluster of near-pad cameras. Lines show the source-to-receiver paths from SLC-40. **B:** enlarged view of SLC-40 and the NE Tower, Pad NW, and Pad West cameras. Camera and launch-pad coordinates were obtained from the project KML metadata, whereas the BCHH position was determined by differential GPS survey. The filled red symbol identifies the FIREY camera supplying the YouTube audio used as an approximate timing aid; open red symbols identify other video locations. Because the video files lack absolute timestamps, the audio alignment was empirical and was not treated as an independently timed quantitative observation.

## 3. Figure 2 — Complete stacked waveform overview


In [ ]:
def robust_scale(values, percentile=99.3):
    values = np.asarray(values, dtype=float)
    scale = np.nanpercentile(np.abs(values[np.isfinite(values)]), percentile)
    return scale if np.isfinite(scale) and scale > 0 else 1.0

def add_time_bar(ax, t0, t1, y, text, above=True):
    x0, x1 = mdates.date2num(t0.datetime), mdates.date2num(t1.datetime)
    ax.plot([x0, x1], [y, y], color="black", lw=1.3, clip_on=False)
    ax.plot([x0, x0], [y - .07, y + .07], color="black", lw=1.3, clip_on=False)
    ax.plot([x1, x1], [y - .07, y + .07], color="black", lw=1.3, clip_on=False)
    ax.text((x0 + x1) / 2, y + (.10 if above else -.10), text,
            ha="center", va="bottom" if above else "top", fontsize=8,
            bbox={"facecolor": "white", "edgecolor": "none", "alpha": .85, "pad": 1})

def stacked_waveform_overview(
    stream,
    start,
    end,
    *,
    channel_order=None,
    phase_bars=None,
    tremor=None,
    event_markers=None,
    extra_traces=None,
    extra_labels=None,
    display_limits_by_channel=None,
    robust_scale_percentile=99.3,
    robust_clip_multiple=3.0,
    maximum_trace_excursion=0.48,
    xlabel="Time on 1 September 2016 (UTC)",
    figsize=(11.5, 5.4),
    use_full_range=False,
):
    display_limits_by_channel = (
        {}
        if display_limits_by_channel is None
        else dict(display_limits_by_channel)
    )

    work = stream.copy().trim(start, end)

    preferred_order = (
        "HD1", "HD2", "HD3",
        "HHE", "HHN", "HHZ",
    )

    if channel_order is None:
        available_channels = {
            trace.stats.channel.upper()
            for trace in work
        }
        channel_order = tuple(
            channel
            for channel in preferred_order
            if channel in available_channels
        )
    else:
        channel_order = tuple(
            channel.upper()
            for channel in channel_order
        )

    traces = []
    labels = []

    for channel in channel_order:
        matches = work.select(channel=channel)

        if len(matches) == 0:
            raise ValueError(
                f"Requested channel {channel!r} is absent. "
                f"Available channels: "
                f"{sorted({tr.stats.channel for tr in work})}"
            )

        if len(matches) > 1:
            raise ValueError(
                f"Expected one {channel} trace, found "
                f"{len(matches)}: {[tr.id for tr in matches]}"
            )

        traces.append(matches[0])
        labels.append(channel)

    if not traces:
        raise ValueError("No requested waveform channels were found.")

    if extra_traces:
        extra = [
            trace.copy().trim(start, end)
            for trace in extra_traces
        ]
        traces = extra + traces
        labels = (
            list(
                extra_labels
                or [trace.stats.channel for trace in extra]
            )
            + labels
        )

    offsets = np.arange(len(traces))[::-1].astype(float)

    fig, ax = plt.subplots(figsize=figsize)

    for offset, trace in zip(offsets, traces):
        channel = trace.stats.channel.upper()

        data = np.asarray(trace.data, dtype=float)
        data = np.nan_to_num(
            data,
            nan=0.0,
            posinf=0.0,
            neginf=0.0,
        )

        if use_full_range:
            data_min = float(np.nanmin(data))
            data_max = float(np.nanmax(data))
            data_range = data_max - data_min

            if not np.isfinite(data_range) or data_range <= 0:
                normalized = np.zeros_like(data)
            else:
                # Map the actual data minimum to -1 and maximum to +1.
                normalized = (
                    2.0 * (data - data_min) / data_range
                    - 1.0
                )

        else:
            physical_limit = display_limits_by_channel.get(channel)

            if physical_limit is not None:
                physical_limit = float(physical_limit)

                if not np.isfinite(physical_limit) or physical_limit <= 0:
                    raise ValueError(
                        f"Invalid display limit for {channel}: "
                        f"{physical_limit}"
                    )

                normalized = np.clip(
                    data / physical_limit,
                    -1.0,
                    1.0,
                )

            else:
                scale = robust_scale(
                    data,
                    percentile=robust_scale_percentile,
                )

                normalized = (
                    np.clip(
                        data / scale,
                        -robust_clip_multiple,
                        robust_clip_multiple,
                    )
                    / robust_clip_multiple
                )

        plotted = (
            offset
            + normalized * maximum_trace_excursion
        )

        ax.plot(
            trace.times("matplotlib"),
            plotted,
            color="black",
            lw=0.42,
            rasterized=True,
        )

    ax.set_yticks(offsets, labels)

    # Separate infrasound and seismic channels when both are present.
    plotted_channels = [
        trace.stats.channel.upper()
        for trace in traces
    ]
    n_seismic = sum(
        channel.startswith("HH")
        for channel in plotted_channels
    )
    n_infrasound = sum(
        channel.startswith("HD")
        for channel in plotted_channels
    )

    if n_seismic and n_infrasound:
        ax.axhline(
            n_seismic - 0.5,
            color="0.55",
            lw=0.7,
        )

    # Separate extra traces, such as audio, from BCHH channels.
    if extra_traces:
        ax.axhline(
            len(traces) - len(extra_traces) - 0.5,
            color="0.55",
            lw=0.7,
        )

    ax.set_ylim(-0.75, len(traces) + 0.42)
    ax.set_xlim(
        mdates.date2num(start.datetime),
        mdates.date2num(end.datetime),
    )

    ax.xaxis.set_major_locator(
        mdates.MinuteLocator(interval=5)
    )
    ax.xaxis.set_major_formatter(
        mdates.DateFormatter("%H:%M")
    )
    ax.xaxis.set_minor_locator(
        mdates.MinuteLocator(interval=1)
    )

    ax.grid(
        axis="x",
        which="major",
        color="0.78",
        lw=0.7,
    )
    ax.grid(
        axis="x",
        which="minor",
        color="0.93",
        lw=0.4,
    )

    ax.set_xlabel(xlabel)

    if phase_bars:
        for name, t0, t1 in phase_bars:
            add_time_bar(
                ax, t0, t1,
                len(traces) + 0.12,
                name,
            )

    if tremor:
        add_time_bar(
            ax,
            tremor[0],
            tremor[1],
            -0.35,
            "tremor",
            above=False,
        )

    if event_markers:
        for name, time in event_markers:
            ax.axvline(
                time.datetime,
                color="tab:red",
                lw=0.8,
                linestyle=":",
            )
            ax.text(
                time.datetime,
                len(traces) - 0.05,
                name,
                color="tab:red",
                rotation=90,
                va="bottom",
                ha="right",
                fontsize=7,
            )

    ax.spines[["top", "right"]].set_visible(False)

    fig.subplots_adjust(
        left=0.075,
        right=0.99,
        top=0.88,
        bottom=0.13,
    )

    return fig, ax

explosion_time = UTCDateTime("2016-09-01T13:07:12.080")
phases = [
    ("Phase I", explosion_time, explosion_time + 70),
    ("Phase II", UTCDateTime("2016-09-01T13:14:15"), UTCDateTime("2016-09-01T13:15:45")),
    ("Phase III", UTCDateTime("2016-09-01T13:19:12"), UTCDateTime("2016-09-01T13:25:00")),
    ("Phase IV", UTCDateTime("2016-09-01T13:29:54"), UTCDateTime("2016-09-01T13:34:54")),
]
fig, ax = stacked_waveform_overview(
    st_baseline, UTCDateTime("2016-09-01T13:05:00"), UTCDateTime("2016-09-01T13:37:00"),
    phase_bars=phases,
    tremor=(UTCDateTime("2016-09-01T13:08:48"), UTCDateTime("2016-09-01T13:12:00")),
)
save_main(fig, "fig02_complete_sequence_waveforms")
plt.show()


> **Figure 2.** Complete BCHH seismo-acoustic record of the approximately 32-minute Falcon 9 explosion sequence. The upper three traces show acoustic pressure recorded by the HD1–HD3 infrasound sensors, and the lower three traces show east, north, and vertical ground velocity recorded by the broadband seismometer. The four brackets identify the principal temporal phases discussed in the text; the interval of sustained seismic tremor following the opening explosion sequence is marked below the HHZ trace. Instrument responses were removed, and a 1 s centered moving-median baseline was subtracted from the infrasound channels. Traces are independently scaled, with extreme excursions limited for display, so amplitudes should not be compared directly between channels. Vertical grid lines mark five-minute intervals.

## 4. Figure 3 — Principal explosion and supplementary key-event waveforms


In [ ]:
# Supplement: compact received-waveform overview of the opening sequence.
youtube_audio_trace = None
if YOUTUBE_AUDIO_FILE.exists():
    numba_cache = PRODUCT_ROOT / "cache" / "numba"
    numba_cache.mkdir(parents=True, exist_ok=True)
    os.environ.setdefault("NUMBA_CACHE_DIR", str(numba_cache))
    import librosa

    audio_values, audio_sampling_rate = librosa.load(YOUTUBE_AUDIO_FILE, sr=None, mono=True)
    youtube_audio_trace = Trace(data=np.asarray(audio_values, dtype=np.float32))
    youtube_audio_trace.stats.network = "YT"
    youtube_audio_trace.stats.station = "SYNC"
    youtube_audio_trace.stats.location = ""
    youtube_audio_trace.stats.channel = "AUD"
    youtube_audio_trace.stats.sampling_rate = float(audio_sampling_rate)
    youtube_audio_trace.stats.starttime = YOUTUBE_AUDIO_STARTTIME
    youtube_audio_trace.detrend("demean")
    youtube_audio_trace.filter(
        "bandpass", freqmin=YOUTUBE_AUDIO_BAND_HZ[0], freqmax=YOUTUBE_AUDIO_BAND_HZ[1],
        corners=4, zerophase=True,
    )
    audio_metadata = pd.DataFrame([{
        "source_file": str(YOUTUBE_AUDIO_FILE),
        "assigned_start_time_utc": str(YOUTUBE_AUDIO_STARTTIME),
        "native_sampling_rate_hz": float(audio_sampling_rate),
        "display_band_min_hz": YOUTUBE_AUDIO_BAND_HZ[0],
        "display_band_max_hz": YOUTUBE_AUDIO_BAND_HZ[1],
        "filter_corners": 4,
        "zero_phase": True,
        "reduced_time_shift_s": YOUTUBE_AUDIO_REDUCED_TIME_SHIFT_S,
        "amplitude_units": "normalized arbitrary amplitude",
        "interpretation": "approximate synchronized video audio, reduced to source time",
    }])
    audio_metadata.to_csv(TABLE_DIR / "youtube_audio_display_metadata.csv", index=False)
else:
    print(f"YouTube audio file unavailable; omitting audio trace: {YOUTUBE_AUDIO_FILE}")

opening_markers = [
    ("upper stage\nexplosion", UTCDateTime("2016-09-01T13:07:12.080")),
    #("Precursor 1", UTCDateTime("2016-09-01T13:07:15.283")),
    #("Precursor 2", UTCDateTime("2016-09-01T13:07:15.517")),
    ("lower stage\nexplosion", UTCDateTime("2016-09-01T13:07:15.750")-0.07),
    ("capsule impact", UTCDateTime("2016-09-01T13:07:24.600")-0.02),
    ("capsule explosion", UTCDateTime("2016-09-01T13:07:25.150")),
]

# Reduce the BCHH airwaves and video audio to the common approximate source-time
# convention used by the archived overview. This is a display transformation only.
opening_stream = st_baseline.copy()
distance_by_channel = geometry.set_index("channel")["distance_m"].to_dict()
correction_speed = 368.8
correction_speed = 366.5
for trace in opening_stream:
    channel = trace.stats.channel.upper()
    if channel in distance_by_channel:
        trace.stats.starttime -= float(distance_by_channel[channel]) / correction_speed

youtube_audio_reduced = None
youtube_time_shift = YOUTUBE_AUDIO_REDUCED_TIME_SHIFT_S - 0.03
if youtube_audio_trace is not None:
    youtube_audio_reduced = youtube_audio_trace.copy()
    youtube_audio_reduced.stats.starttime -= youtube_time_shift


plotted_stream = opening_stream.copy()

fig, ax = stacked_waveform_overview(
    plotted_stream,
    UTCDateTime("2016-09-01T13:07:10.5"),
    UTCDateTime("2016-09-01T13:07:27.5"),
    event_markers=opening_markers,
    extra_traces=None if youtube_audio_reduced is None else [youtube_audio_reduced],
    extra_labels=None if youtube_audio_reduced is None else ["YouTube audio\n1–2 kHz"],
    xlabel="Approximate source-reduced time on 1 September 2016 (UTC)",
    figsize=(11.5, 5.0),
)
ax.xaxis.set_major_locator(mdates.SecondLocator(interval=2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M:%S"))
save_supplement(fig, "figS01_opening_sequence_waveforms")
plt.show()

> **Figure S1.** Approximate source-time comparison of public-video audio and the complete BCHH seismo-acoustic record during the opening explosion sequence. The upper trace shows normalized YouTube audio band-pass filtered between 1 and 2 kHz with a four-corner, zero-phase Butterworth filter. The next three traces show acoustic pressure on HD1–HD3, and the lower three show east, north, and vertical ground velocity on HHE, HHN, and HHZ. BCHH records were shifted by channel-specific travel times using an apparent propagation speed of 366.5 m s\(^{-1}\); the untimestamped audio was aligned empirically. Red dotted lines mark the best-estimate times of the upper-stage explosion, lower-stage explosion, capsule impact, and capsule explosion. The upper-stage onset is directly identifiable in the video, whereas the remaining times represent approximate interpretations of the video chronology. Instrument responses were removed, and a 1 s centered moving-median baseline was subtracted from the infrasound channels. Traces are independently scaled for display; amplitudes should not be compared between channels.

In [ ]:
#different filtered version to line up sound and infrasound
plotted_stream2 = plotted_stream.copy().filter(type='bandpass', freqmin=30, freqmax=120)
fig, ax = stacked_waveform_overview(
    plotted_stream2,
    UTCDateTime("2016-09-01T13:07:10.5"),
    UTCDateTime("2016-09-01T13:07:27.5"),
    event_markers=opening_markers,
    extra_traces=None if youtube_audio_reduced is None else [youtube_audio_reduced],
    extra_labels=None if youtube_audio_reduced is None else ["YouTube audio\n1–2 kHz"],
    xlabel="Approximate source-reduced time on 1 September 2016 (UTC)",
    figsize=(11.5, 5.0),
)
ax.xaxis.set_major_locator(mdates.SecondLocator(interval=2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M:%S"))


In [ ]:
fig, ax = stacked_waveform_overview(
    opening_stream,
    UTCDateTime("2016-09-01T13:07:10.5"),
    UTCDateTime("2016-09-01T13:07:27.5"),
    channel_order=("HD2", "HHZ"),
    event_markers=opening_markers,
    extra_traces=(
        None
        if youtube_audio_reduced is None
        else [youtube_audio_reduced]
    ),
    extra_labels=(
        None
        if youtube_audio_reduced is None
        else ["YouTube\naudio"]
    ),
    use_full_range=True,
    maximum_trace_excursion=0.48,
    xlabel=(
        "Approximate source-reduced time on "
        "1 September 2016 (UTC)"
    ),
    figsize=(11.5, 8.8),
)

ax.xaxis.set_major_locator(
    mdates.SecondLocator(interval=2)
)
ax.xaxis.set_major_formatter(
    mdates.DateFormatter("%H:%M:%S")
)

ax.set_yticklabels([
    "YouTube\naudio",
    "Infrasound\n(HD2)",
    "Seismic\n(HHZ)",
])
save_supplement(fig, "fig03_phase1")
plt.show()

**Figure 3.** Source-time-reduced audio, infrasound, and seismic waveforms during the opening explosion sequence. The upper trace shows 1–2 kHz audio extracted from an untimestamped YouTube recording and empirically synchronized with the BCHH observations. The middle and lower traces show acoustic pressure on HD2 and vertical ground velocity on HHZ, respectively, shifted by their estimated propagation times to approximate source time. Red dotted lines mark the best-estimate times of the upper-stage explosion, lower-stage explosion, capsule impact, and capsule explosion. The upper-stage onset is directly identifiable in the video; the remaining times are approximate interpretations based on the video chronology. Each trace is independently scaled to its observed minimum and maximum over the displayed interval, without clipping; amplitudes should therefore not be compared between traces.

In [ ]:
# Main figure: give the principal explosion enough space for each physical channel.
principal_start = UTCDateTime("2016-09-01T13:07:18.000")
principal_stream = st_baseline.copy().trim(principal_start, principal_start + 3.0)
fig, axes = plot_key_event_waveforms(
    principal_stream,
    reference_time=principal_start,
    pressure_results=None,
    title="Principal Falcon 9 explosion",
    add_measurements=True,
)
save_main(fig, "fig04_principal_explosion_waveforms")
plt.show()

# Supplement: the weaker initial failure and the later two-pulse capsule sequence.
second_start = UTCDateTime("2016-09-01T13:07:15.000")
second_stream = st_baseline.copy().trim(second_start, second_start + 2.0)
fig, axes = plot_key_event_waveforms(
    second_stream, reference_time=second_start, pressure_results=None,
    title="Initial second-stage failure", add_measurements=True,
)
save_supplement(fig, "figS02_second_stage_waveforms")
plt.show()

capsule_start = UTCDateTime("2016-09-01T13:07:27.000")
capsule_stream = st_baseline.copy().trim(capsule_start, capsule_start + 3.0)
fig, axes = plot_key_event_waveforms(
    capsule_stream, reference_time=capsule_start, pressure_results=None,
    title="Capsule-related acoustic pulses", add_measurements=True,
)
save_supplement(fig, "figS03_capsule_waveforms")
plt.show()


**Figure 4 — Principal explosion**

> **Figure 4.** Calibrated seismic and infrasound waveforms associated with the principal Falcon 9 explosion. The left column shows the vertical, north, and east components of ground velocity, and the right column shows acoustic pressure recorded by HD1–HD3. Time is relative to 13:07:18.000 UTC at BCHH. Instrument responses were removed and a 1 s centered moving-median baseline was subtracted from the infrasound records. Markers identify the measured positive and negative pressure extrema. All three infrasound sensors recorded a sharp positive compression followed by a smaller rarefaction and gradual recovery, producing a strongly asymmetric, Friedlander-like pressure waveform. The seismic channels record the corresponding ground-coupled airwave and subsequent surface-wave motion.

**Figure S2 — Initial second-stage failure**

> **Figure S2.** Calibrated seismic and infrasound waveforms associated with the initial second-stage failure. The left column shows three-component ground velocity, and the right column shows acoustic pressure on HD1–HD3. Time is relative to 13:07:15.000 UTC at BCHH. Infrasound baselines were removed using a 1 s centered moving median, and markers identify the measured positive and negative pressure extrema. This initial arrival was substantially weaker and less coherent in waveform shape than the subsequent principal explosion. The seismic signals represent ground motion generated primarily by coupling of the arriving airwave.

**Figure S3 — Capsule-related pulses**

> **Figure S3.** Calibrated seismic and infrasound waveforms associated with the capsule-related sequence. The left column shows three-component ground velocity, and the right column shows acoustic pressure on HD1–HD3. Time is relative to 13:07:27.000 UTC at BCHH. Two coherent acoustic pulses separated by approximately 0.5–0.6 s are visible across the infrasound array, with corresponding ground-coupled airwaves on the seismometer. Markers identify the measured positive and negative pressure extrema. Comparison with the video chronology suggests that the first pulse may be associated with capsule impact and the second with a subsequent explosion or rupture, but the physical origins cannot be established from the waveform data alone.

## 5. Figure 5 — Planar coherence and catalogue array results


In [ ]:
principal_surface = np.load(
    PLANAR_ROOT / "surfaces" / f"event_{PRINCIPAL_EVENT_NUMBER:03d}_planar_surface.npz"
)
composite = np.load(PLANAR_ROOT / "composites" / "composite_core_catalogue.npz")
expected_baz = float(analysis_config["array_reference"]["back_azimuth_to_slc40_deg"])
weather_speed = float(pd.read_csv(
    PROJECT_ROOT / "outputs" / "weather" / "derived" / "weather_acoustic_summary.csv"
).iloc[0]["effective_sound_speed_mps"])

fig = plt.figure(figsize=(13, 10), constrained_layout=True)
grid = fig.add_gridspec(2, 2)
ax0 = fig.add_subplot(grid[0, 0], projection="polar")
ax1 = fig.add_subplot(grid[0, 1], projection="polar")
ax2 = fig.add_subplot(grid[1, 0])
ax3 = fig.add_subplot(grid[1, 1])

def polar_surface(ax, azimuth, speed, score, title):
    mesh = ax.pcolormesh(np.deg2rad(azimuth), speed, score.T, shading="auto", cmap="viridis")
    ax.set_theta_zero_location("N")
    ax.set_theta_direction(-1)
    best = np.unravel_index(np.nanargmax(score), score.shape)
    ax.plot(np.deg2rad(azimuth[best[0]]), speed[best[1]], "*", ms=13,
            color="white", mec="black")
    ax.plot(np.deg2rad(expected_baz), weather_speed, "o", ms=6,
            color="orange", mec="black")
    ax.set_title(title, pad=16)
    return mesh

mesh0 = polar_surface(
    ax0, principal_surface["fine_back_azimuth_deg"],
    principal_surface["fine_apparent_speed_mps"],
    principal_surface["fine_coherence_score"], "Principal explosion",
)
mesh1 = polar_surface(
    ax1, composite["back_azimuth_deg"], composite["apparent_speed_mps"],
    composite["normalized_mean"], "Core-event composite",
)
fig.colorbar(mesh0, ax=ax0, pad=0.12, label="Coherence score")
fig.colorbar(mesh1, ax=ax1, pad=0.12, label="Normalized coherence")

planar_plot = planar.merge(
    measurements[["event_number", "elapsed_time_min"]], on="event_number", how="left"
)
core = planar_plot["passes_core_filter"].fillna(False).astype(bool)
ax2.scatter(planar_plot.loc[~core, "elapsed_time_min"], planar_plot.loc[~core, "best_back_azimuth_deg"],
            s=18, color="0.65", alpha=0.55, label="Additional analyzed")
ax2.scatter(planar_plot.loc[core, "elapsed_time_min"], planar_plot.loc[core, "best_back_azimuth_deg"],
            s=25, color="tab:blue", alpha=0.85, label="Core")
ax2.axhline(expected_baz, color="orange", ls="--", lw=1, label="SLC-40")
ax2.set_xlabel("Time since first candidate arrival (min)")
ax2.set_ylabel("Back azimuth (°)")
ax2.grid(True, alpha=0.2)
ax2.legend()

ax3.scatter(planar_plot.loc[~core, "elapsed_time_min"], planar_plot.loc[~core, "best_apparent_speed_mps"],
            s=18, color="0.65", alpha=0.55)
ax3.scatter(planar_plot.loc[core, "elapsed_time_min"], planar_plot.loc[core, "best_apparent_speed_mps"],
            s=25, color="tab:blue", alpha=0.85)
ax3.axhline(weather_speed, color="orange", ls="--", lw=1)
ax3.set_xlabel("Time since first candidate arrival (min)")
ax3.set_ylabel("Apparent speed (m s$^{-1}$)")
ax3.grid(True, alpha=0.2)

save_main(fig, "fig05_planar_array_results")
plt.show()


> **Figure 5.** Plane-wave array analysis of the Falcon 9 explosion sequence. **Top left:** joint three-channel coherence surface for the principal explosion as a function of back azimuth (polar angle, clockwise from north) and apparent horizontal speed (radial coordinate, m s\(^{-1}\)). **Top right:** normalized mean coherence surface obtained by combining the core catalogue events that passed the amplitude, signal-to-noise, and array-quality criteria. White stars mark the maximum-coherence solutions; orange circles indicate the expected back azimuth to SLC-40 and the wind-corrected acoustic speed derived from meteorological observations. **Bottom left:** best-fitting back azimuth for each analyzed event as a function of time since the first candidate arrival. **Bottom right:** corresponding apparent horizontal speeds. Blue symbols denote core events and gray symbols denote additional analyzed candidates; orange dashed lines show the expected direction to SLC-40 and the weather-derived effective sound speed. Most core events form a compact population close to the launch-complex direction and expected acoustic velocity, whereas lower-quality candidates exhibit substantially greater scatter.

## 6. Figure 6 — Acoustic and seismic amplitudes through time


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True, constrained_layout=True)
columns = [
    ("acoustic_stack_peak_to_peak_pa", "Stack P2P pressure (Pa)"),
    ("pgv_vector_mps", "Vector PGV (m s$^{-1}$)"),
    ("p2p_pressure_to_vector_pgv", "P2P pressure / PGV\n(Pa per m s$^{-1}$)"),
]
for ax, (column, label) in zip(axes, columns):
    for tier, style in TIER_STYLE.items():
        subset = measurements.loc[measurements["selection_tier"].eq(tier)]
        ax.scatter(subset["elapsed_time_min"], subset[column], edgecolor="none", **style)
    ax.set_yscale("log")
    ax.set_ylabel(label)
    ax.grid(True, which="both", alpha=0.2)
principal = measurements.loc[measurements["event_number"].eq(PRINCIPAL_EVENT_NUMBER)].iloc[0]
for ax in axes:
    ax.axvline(principal["elapsed_time_min"], color="orange", ls="--", lw=1)
#axes[0].set_title("Event amplitudes and acoustic–seismic coupling through time")
axes[0].legend(ncol=3)
axes[-1].set_xlabel("Time since first candidate arrival (min)")
save_main(fig, "fig06_amplitudes_and_coupling_vs_time")
plt.show()


> **Figure 6.** Evolution of event amplitude and acoustic–seismic coupling during the approximately 26-minute explosion sequence. **Top:** median-stack peak-to-peak acoustic pressure measured across the three infrasound sensors. **Middle:** peak three-component vector ground velocity (PGV) measured over the corresponding event window. **Bottom:** ratio of peak-to-peak acoustic pressure to vector PGV, used as an amplitude-based proxy for acoustic–seismic coupling. Blue circles denote the core planar-array event set, green triangles denote additional analyzed events, and small gray symbols denote other candidates. The orange dashed line marks the principal explosion. All vertical axes are logarithmic. The largest acoustic and seismic amplitudes occurred during the opening sequence, followed by generally weaker activity during later phases; the pressure-to-PGV ratio characterizes variations in relative atmospheric and ground response but is not an energy ratio.

## 7. Figure 7 — Acoustic pressure versus vector PGV


In [ ]:
valid = measurements.loc[
    np.isfinite(measurements["acoustic_stack_peak_to_peak_pa"])
    & (measurements["acoustic_stack_peak_to_peak_pa"] > 0)
    & np.isfinite(measurements["pgv_vector_mps"])
    & (measurements["pgv_vector_mps"] > 0)
].copy()

fig, ax = plt.subplots(figsize=(8.5, 7), constrained_layout=True)
for tier, style in TIER_STYLE.items():
    subset = valid.loc[valid["selection_tier"].eq(tier)]
    ax.scatter(subset["pgv_vector_mps"], subset["acoustic_stack_peak_to_peak_pa"],
               edgecolor="none", **style)

xline = np.logspace(np.log10(valid["pgv_vector_mps"].min() / 1.4),
                    np.log10(valid["pgv_vector_mps"].max() * 1.4), 300)
for ratio in (1e5, 1e6, 1e7, 1e8):
    ax.plot(xline, ratio * xline, color="0.65", lw=0.8, ls="--", zorder=0)

quality = valid.loc[valid["passes_regression_quality"].fillna(False).astype(bool)]
if len(quality) >= 3:
    '''
    fit = linregress(np.log10(quality["pgv_vector_mps"]),
                     np.log10(quality["acoustic_stack_peak_to_peak_pa"]))
    ax.plot(xline, 10 ** (fit.intercept + fit.slope * np.log10(xline)),
            color="tab:red", lw=2, label=f"Linear fit: slope {fit.slope:.2f}")
    '''
    ax.scatter(quality["pgv_vector_mps"], quality["acoustic_stack_peak_to_peak_pa"],
               facecolor="none", edgecolor="black", s=50, lw=0.7,
               label="Regression subset")
    
log_x = np.log10(quality["pgv_vector_mps"].to_numpy())
log_y = np.log10(
    quality["acoustic_stack_peak_to_peak_pa"].to_numpy()
)

# Unconstrained power law: log10(P) = intercept + slope * log10(PGV)
fit = linregress(log_x, log_y)
unconstrained_y = 10 ** (
    fit.intercept + fit.slope * np.log10(xline)
)

# Proportional model: P = K * PGV, equivalent to fixing log-log slope at 1.
log10_k = np.mean(log_y - log_x)
k = 10 ** log10_k
proportional_y = k * xline

ax.plot(
    xline,
    unconstrained_y,
    color="tab:red",
    lw=2,
    label=f"Power-law fit",
)
ax.plot(
    xline,
    proportional_y,
    color="tab:purple",
    lw=1.8,
    ls="--",
    label=fr"Proportional fit",
)

# Compare log-space residual errors.
predicted_free = fit.intercept + fit.slope * log_x
predicted_proportional = log10_k + log_x

rmse_free = np.sqrt(np.mean((log_y - predicted_free) ** 2))
rmse_proportional = np.sqrt(
    np.mean((log_y - predicted_proportional) ** 2)
)

print(f"Unconstrained exponent: {fit.slope:.3f}")
print(f"Proportionality constant K: {k/1e6:.3f} MPa per (m/s)")
print(f"Free power-law log10 RMSE: {rmse_free:.3f}")
print(f"Proportional-model log10 RMSE: {rmse_proportional:.3f}")

principal = valid.loc[valid["event_number"].eq(PRINCIPAL_EVENT_NUMBER)].iloc[0]
ax.scatter(principal["pgv_vector_mps"], principal["acoustic_stack_peak_to_peak_pa"],
           marker="*", s=230, facecolor="orange", edgecolor="black",
           zorder=6, label="Principal explosion")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_ylim(7, 2e3)
ax.set_xlim(3e-6, 7e-3 )
ax.set_xlabel("Three-component vector PGV (m s$^{-1}$)")
ax.set_ylabel("Median-stack peak-to-peak pressure (Pa)")
#ax.set_title("Acoustic pressure versus ground velocity")
ax.grid(True, which="both", alpha=0.2)
ax.legend()
save_main(fig, "fig07_pressure_vs_vector_pgv")
plt.show()


**Figure 7.** Relationship between acoustic and seismic amplitudes for candidate events. Median-stack peak-to-peak acoustic pressure is plotted against three-component vector peak ground velocity (PGV) on logarithmic axes. Blue circles denote the core planar-array set, green triangles denote additional analyzed events, and small gray points denote other candidates; black open circles identify the regression-quality subset. The red line is the best-fitting power law, \(p_{\mathrm{p2p}}\propto\mathrm{PGV}^{0.868}\), with a log-space RMSE of 0.128 dex. The purple dashed line is the proportional model \(p_{\mathrm{p2p}}=K\,\mathrm{PGV}\), for which \(K=0.447\ \mathrm{MPa}\,(\mathrm{m\,s^{-1}})^{-1}\) and the RMSE is 0.146 dex. Gray dashed diagonals indicate constant pressure-to-PGV ratios, and the principal explosion is marked by the orange star. The similar errors of the two models indicate that approximately proportional acoustic–seismic amplitude scaling provides a reasonable description, although the freely fitted sublinear relationship performs slightly better.

I would use “log-space RMSE” and “multiplicative factor” in the paper, omitting “dex” entirely. Dex is useful shorthand, but it adds vocabulary without adding much here.

## 8. Supplementary Figure S4 — Spherical-wave constraints


In [ ]:
profile_npz = SPHERICAL_ROOT / "surfaces" / f"event_{PRINCIPAL_EVENT_NUMBER:03d}_range_profiles.npz"
profile = np.load(profile_npz)

fig, axes = plt.subplots(1, 3, figsize=(16, 5), constrained_layout=True)
axes[0].scatter(spherical["planar_speed_mps"], spherical["fixed_source_speed_mps"],
                c=spherical["fixed_source_coherence_score"], cmap="viridis", s=25)
limits = [min(spherical["planar_speed_mps"].min(), spherical["fixed_source_speed_mps"].min()),
          max(spherical["planar_speed_mps"].max(), spherical["fixed_source_speed_mps"].max())]
axes[0].plot(limits, limits, color="0.4", ls="--")
axes[0].set_xlabel("Planar apparent speed (m s$^{-1}$)")
axes[0].set_ylabel("Fixed-SLC-40 spherical speed (m s$^{-1}$)")
axes[0].grid(True, alpha=0.2)

mesh = axes[1].pcolormesh(profile["cross_range_m"], profile["along_range_m"],
                          profile["event_speed_score"], shading="auto", cmap="viridis")
axes[1].axhline(0, color="white", lw=0.8, ls="--")
axes[1].axvline(0, color="white", lw=0.8, ls="--")
axes[1].set_xlabel("Cross-range offset from SLC-40 (m)")
axes[1].set_ylabel("Along-range offset from SLC-40 (m)")
axes[1].set_title("Principal-explosion coherence surface")
fig.colorbar(mesh, ax=axes[1], label="Coherence score")

axes[2].plot(profile["cross_range_m"], profile["cross_profile_event_speed"],
             label="Cross-range profile")
axes[2].plot(profile["along_range_m"], profile["along_profile_event_speed"],
             label="Along-range profile")
axes[2].set_xlabel("Offset (m)")
axes[2].set_ylabel("Maximum coherence")
axes[2].set_title("Open range versus finite cross-range constraint")
axes[2].grid(True, alpha=0.2)
axes[2].legend()
save_supplement(fig, "figS04_spherical_constraints")
plt.show()


> **Figure Sx.** Comparison of planar and spherical-wave array solutions and assessment of source-location resolution. **Left:** apparent speeds obtained from the planar-wave analysis compared with spherical-wave speeds calculated for a source fixed at SLC-40; the dashed line denotes equality. **Center:** coherence surface for the principal explosion as a function of cross-range and along-range displacement from SLC-40. Dashed lines mark the launch-pad position. **Right:** maximum-coherence profiles through the two-dimensional surface. Coherence exhibits a distinct cross-range maximum but varies very little with along-range position. The close correspondence between planar and fixed-source spherical speeds, together with the broad along-range coherence ridge, shows that the BCHH array constrains arrival direction more effectively than source range. Consequently, unconstrained spherical locations along the source–array direction should not be interpreted as precise source positions.

I would include this if the manuscript discusses why spherical localization was investigated and ultimately treated cautiously. Otherwise, it could be condensed into a supplementary figure and a short Methods/Discussion paragraph.

## 9. Table 1 — Instrumentation and calibration


In [ ]:
table1 = pd.DataFrame([
    {"Instrument": "Nanometrics Trillium Compact Posthole", "Channels": "HHZ, HHN, HHE",
     "Sampling rate (Hz)": 250, "Calibration": "3.02 × 10^8 counts (m s^-1)^-1"},
    {"Instrument": "infraBSU", "Channels": "HD1", "Sampling rate (Hz)": 250,
     "Calibration": "8.10 counts Pa^-1"},
    {"Instrument": "infraBSU", "Channels": "HD2", "Sampling rate (Hz)": 250,
     "Calibration": "0.690 counts Pa^-1"},
    {"Instrument": "infraBSU", "Channels": "HD3", "Sampling rate (Hz)": 250,
     "Calibration": "10.0 counts Pa^-1"},
])
table1.to_csv(TABLE_DIR / "table01_instrumentation_calibration.csv", index=False)
(TABLE_DIR / "table01_instrumentation_calibration.tex").write_text(
    table1.to_latex(index=False, escape=False)
)
display(table1)


**Table 1.** Instrumentation, channel assignments, sampling rates, and calibration factors used for waveform processing at BCHH. All channels were digitized continuously at 250 samples s\(^{-1}\) by a Nanometrics Centaur operated with a 40 V peak-to-peak input range and sensitivity of \(4.0\times10^{5}\) counts V\(^{-1}\). The Trillium Compact manufacturer response of \(3.02\times10^{8}\) counts (m s\(^{-1}\))\(^{-1}\) corresponds to approximately 754 V (m s\(^{-1}\))\(^{-1}\). The infraBSU values are sensor-specific empirical field calibrations used to convert digital counts to acoustic pressure; these are approximate rather than laboratory-traceable absolute calibrations.

## 10. Table 2 — Key stages of the accident


In [ ]:
# Canonical identifiers used throughout the final paper products.
EVENT_ID_ALIASES = {
    "capsule_pulse_1": "capsule_impact",
    "capsule_pulse_2": "capsule_explosion",
}

KEY_EVENT_NUMBERS = {
    "second_stage": 2,
    "principal_explosion": 13,
    "capsule_impact": 22,
    "capsule_explosion": 23,
}

KEY_EVENT_DISPLAY = {
    "second_stage": "Initial second-stage failure",
    "principal_explosion": "Principal explosion",
    "capsule_impact": "Capsule impact",
    "capsule_explosion": "Capsule explosion",
}


# Standardize legacy identifiers in the input table.
key_events_table = key_events.copy()
key_events_table["event_id"] = (
    key_events_table["event_id"]
    .astype(str)
    .replace(EVENT_ID_ALIASES)
)

expected_ids = set(KEY_EVENT_NUMBERS)
observed_ids = set(key_events_table["event_id"])

missing_ids = expected_ids - observed_ids
unexpected_ids = observed_ids - expected_ids

if missing_ids or unexpected_ids:
    raise ValueError(
        "Unexpected key-event identifiers after canonicalization. "
        f"Missing: {sorted(missing_ids)}; "
        f"unexpected: {sorted(unexpected_ids)}"
    )

if key_events_table["event_id"].duplicated().any():
    duplicates = key_events_table.loc[
        key_events_table["event_id"].duplicated(keep=False),
        "event_id",
    ].tolist()
    raise ValueError(
        f"Duplicate key-event definitions: {duplicates}"
    )


source_distance_m = float(
    analysis_config["array_reference"]["distance_m"]
)

key_rows = []

for event in key_events_table.itertuples(index=False):
    event_id = event.event_id
    event_number = KEY_EVENT_NUMBERS[event_id]

    amplitude_match = measurements.loc[
        measurements["event_number"].eq(event_number)
    ]

    if len(amplitude_match) != 1:
        raise ValueError(
            f"Expected one amplitude row for event {event_number} "
            f"({event_id}), found {len(amplitude_match)}."
        )

    amplitude = amplitude_match.iloc[0]

    array_match = planar.loc[
        planar["event_number"].eq(event_number)
    ]

    if len(array_match) > 1:
        raise ValueError(
            f"Expected no more than one planar-array result for "
            f"event {event_number}, found {len(array_match)}."
        )

    array_row = (
        array_match.iloc[0]
        if len(array_match) == 1
        else None
    )

    positive_pressure_pa = float(
        amplitude["acoustic_stack_positive_peak_pa"]
    )

    key_rows.append({
        "Stage": KEY_EVENT_DISPLAY[event_id],
        "Candidate event": event_number,
        "Source time (UTC)": event.source_time,
        "Positive pressure (Pa)": positive_pressure_pa,
        "Negative pressure (Pa)": float(
            amplitude["acoustic_stack_negative_peak_pa"]
        ),
        "Peak-to-peak pressure (Pa)": float(
            amplitude["acoustic_stack_peak_to_peak_pa"]
        ),
        "Reduced positive pressure at 1 km (Pa)": (
            positive_pressure_pa
            * source_distance_m
            / 1000.0
        ),
        "Vector PGV (m s^-1)": float(
            amplitude["pgv_vector_mps"]
        ),
        "Back azimuth (deg)": (
            np.nan
            if array_row is None
            else float(array_row["best_back_azimuth_deg"])
        ),
        "Planar apparent speed (m s^-1)": (
            np.nan
            if array_row is None
            else float(array_row["best_apparent_speed_mps"])
        ),
    })


table2 = pd.DataFrame(key_rows)

# Format source times to millisecond precision.
source_times = pd.to_datetime(
    table2["Source time (UTC)"],
    utc=True,
)

table2["Source time (UTC)"] = (
    source_times.dt.strftime("%Y-%m-%dT%H:%M:%S.%f")
    .str.slice(stop=-3)
    + "Z"
)

# Round pressures to the nearest pascal.
pressure_columns = [
    "Positive pressure (Pa)",
    "Negative pressure (Pa)",
    "Peak-to-peak pressure (Pa)",
    "Reduced positive pressure at 1 km (Pa)",
]

for column in pressure_columns:
    table2[column] = (
        table2[column]
        .round(0)
        .astype("Int64")
    )

# Retain three significant figures for PGV.
table2["Vector PGV (m s^-1)"] = table2[
    "Vector PGV (m s^-1)"
].map(
    lambda value: (
        np.nan
        if not np.isfinite(value)
        else float(f"{value:.3g}")
    )
)

# Array quantities are limited by the search-grid resolution.
table2["Back azimuth (deg)"] = table2[
    "Back azimuth (deg)"
].round(1)

table2["Planar apparent speed (m s^-1)"] = table2[
    "Planar apparent speed (m s^-1)"
].round(1)

table2.to_csv(
    TABLE_DIR / "table02_key_accident_stages.csv",
    index=False,
)

(
    TABLE_DIR / "table02_key_accident_stages.tex"
).write_text(
    table2.to_latex(index=False),
    encoding="utf-8",
)

display(table2)


The failure is because `key_events.csv` still contains `capsule_pulse_1` and `capsule_pulse_2`. The previous code renamed dictionary keys but never renamed the identifiers in `key_events` itself.

Suggested caption:

**Table 2.** Summary measurements for four key stages of the Falcon 9 accident sequence. Source times are best estimates derived from the video chronology. Acoustic amplitudes are measured from the median stack of the three calibrated, baseline-corrected infrasound channels; reduced positive pressures are scaled to a reference distance of 1 km assuming spherical \(1/r\) spreading from the BCHH array-reference distance. Seismic amplitude is represented by the peak three-component vector ground velocity within the corresponding event window. Back azimuth and apparent speed are obtained from the planar-wave coherence analysis; missing values indicate events for which no accepted array solution was available. Pressure amplitudes carry an estimated absolute calibration uncertainty of approximately ±20%.

## 11. Table 3 — Principal explosion summary


In [ ]:
principal_amp = measurements.loc[
    measurements["event_number"].eq(PRINCIPAL_EVENT_NUMBER)
].iloc[0]
principal_planar = planar.loc[planar["event_number"].eq(PRINCIPAL_EVENT_NUMBER)].iloc[0]
principal_spherical = spherical.loc[
    spherical["event_number"].eq(PRINCIPAL_EVENT_NUMBER)
].iloc[0]
principal_source_time = key_events.loc[
    key_events["event_id"].eq("principal_explosion"), "source_time"
].iloc[0]
reference_distance = float(analysis_config["array_reference"]["distance_m"])
positive_pressure = float(principal_amp["acoustic_stack_positive_peak_pa"])

principal_values = [
    ("Estimated source time", principal_source_time, "UTC"),
    ("Positive pressure", positive_pressure, "Pa"),
    ("Negative pressure", principal_amp["acoustic_stack_negative_peak_pa"], "Pa"),
    ("Peak-to-peak pressure", principal_amp["acoustic_stack_peak_to_peak_pa"], "Pa"),
    ("Reduced positive pressure", positive_pressure * reference_distance / 1000.0, "Pa at 1 km"),
    ("Peak sound-pressure level", 20 * np.log10(positive_pressure / 20e-6), "dB re 20 µPa"),
    ("Three-component vector PGV", principal_amp["pgv_vector_mps"], "m s^-1"),
    ("Planar back azimuth", principal_planar["best_back_azimuth_deg"], "deg"),
    ("Planar apparent speed", principal_planar["best_apparent_speed_mps"], "m s^-1"),
    ("Planar coherence score", principal_planar["maximum_coherence_score"], "dimensionless"),
    ("Fixed-SLC-40 spherical speed", principal_spherical["fixed_source_speed_mps"], "m s^-1"),
]

energy_file = DERIVED_DIR / "principal_explosion_energy_summary.csv"
if energy_file.exists():
    energy = pd.read_csv(energy_file).iloc[0]
    optional = [
        ("Acoustic energy", energy.get("acoustic_energy_j", np.nan), "J"),
        ("Seismic energy", energy.get("seismic_energy_j", np.nan), "J"),
        ("Equivalent acoustic magnitude", energy.get("acoustic_magnitude", np.nan), ""),
        ("Equivalent seismic magnitude", energy.get("seismic_magnitude", np.nan), ""),
    ]
    principal_values.extend(optional)
else:
    print("Energy rows omitted: principal_explosion_energy_summary.csv is not yet available.")

table3 = pd.DataFrame(principal_values, columns=["Parameter", "Value", "Units"])
def format_principal_value(row):
    parameter, value = row["Parameter"], row["Value"]
    if parameter == "Estimated source time":
        return pd.to_datetime(value, utc=True).strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3] + "Z"
    value = float(value)
    if "pressure" in parameter.lower() and "level" not in parameter.lower():
        return f"{value:.0f}"
    if parameter == "Peak sound-pressure level":
        return f"{value:.1f}"
    if parameter == "Three-component vector PGV":
        return f"{value:.3g}"
    if parameter in {"Planar back azimuth", "Planar apparent speed",
                     "Fixed-SLC-40 spherical speed"}:
        return f"{value:.1f}"
    if parameter == "Planar coherence score":
        return f"{value:.3f}"
    if "energy" in parameter.lower():
        return f"{value:.3g}"
    return f"{value:.2f}"

table3["Value"] = table3.apply(format_principal_value, axis=1)
table3.to_csv(TABLE_DIR / "table03_principal_explosion_summary.csv", index=False)
(TABLE_DIR / "table03_principal_explosion_summary.tex").write_text(
    table3.to_latex(index=False, escape=False)
)
display(table3)


**Table 3.** Summary parameters for the principal Falcon 9 explosion. The source time is the best estimate derived from the synchronized video chronology. Positive, negative, and peak-to-peak pressures are measured from the median stack of the three calibrated, baseline-corrected infrasound channels. Reduced positive pressure is scaled to 1 km assuming spherical \(1/r\) spreading from the BCHH array-reference distance. Peak sound-pressure level is calculated from the positive pressure maximum using a reference pressure of \(20\ \mu\mathrm{Pa}\) and is distinct from a time-averaged RMS sound-pressure level. Seismic amplitude is represented by the peak three-component vector ground velocity. Back azimuth, apparent speed, and coherence are obtained from the planar-wave analysis; the spherical speed assumes a source fixed at SLC-40. Acoustic and seismic energies and their equivalent magnitudes, where available, are first-order estimates. Absolute pressure amplitudes have an estimated calibration uncertainty of approximately ±20%.

## 12. Table 4 — Catalogue and array-analysis summary


In [ ]:
core_planar = planar.loc[planar["passes_core_filter"].fillna(False).astype(bool)]
summary_rows = [
    ("Multi-channel candidate events", len(candidates), "count"),
    ("Events passing permissive planar filter", int(decisions["passes_analysis_filter"].sum()), "count"),
    ("Core planar events", int(decisions["passes_core_filter"].sum()), "count"),
    ("Median planar back azimuth", planar["best_back_azimuth_deg"].median(), "deg"),
    ("Planar back-azimuth IQR", planar["best_back_azimuth_deg"].quantile(.75) - planar["best_back_azimuth_deg"].quantile(.25), "deg"),
    ("Median planar apparent speed", planar["best_apparent_speed_mps"].median(), "m s^-1"),
    ("Planar apparent-speed IQR", planar["best_apparent_speed_mps"].quantile(.75) - planar["best_apparent_speed_mps"].quantile(.25), "m s^-1"),
    ("Median planar coherence", planar["maximum_coherence_score"].median(), "dimensionless"),
    ("Median fixed-source spherical speed", spherical["fixed_source_speed_mps"].median(), "m s^-1"),
    ("Events in high-quality coupling regression", int(measurements["passes_regression_quality"].sum()), "count"),
]
table4 = pd.DataFrame(summary_rows, columns=["Quantity", "Value", "Units"])
def format_summary_value(row):
    value = float(row["Value"])
    if row["Units"] == "count":
        return f"{int(round(value))}"
    if row["Units"] in {"deg", "m s^-1"}:
        return f"{value:.1f}"
    if row["Units"] == "dimensionless":
        return f"{value:.3f}"
    return f"{value:.3g}"

table4["Value"] = table4.apply(format_summary_value, axis=1)
table4.to_csv(TABLE_DIR / "table04_catalogue_array_summary.csv", index=False)
(TABLE_DIR / "table04_catalogue_array_summary.tex").write_text(
    table4.to_latex(index=False, escape=False)
)
display(table4)


**Table 4.** Summary statistics for the candidate-event catalogue and array analyses. The table reports the number of multi-channel candidate events, the subsets satisfying the configurable permissive and core planar-analysis filters, and the number retained for the high-quality acoustic–seismic coupling regression. Median back azimuth, apparent speed, and coherence are calculated from the available planar-wave solutions; interquartile ranges (IQRs) are the differences between the 75th and 25th percentiles. The fixed-source spherical result is the median speed obtained when sources are constrained to SLC-40.

One code detail: `core_planar` is defined but not used, so the reported planar medians and IQRs currently describe **all rows in `planar`**, not only the core events. If the table is intended to summarize the core set, change those expressions to `core_planar[...]`.

## 13. Supplementary catalogue tables and product manifest


In [ ]:
supplement_catalogue = (
    measurements.merge(
        planar[[c for c in planar.columns if c != "stream_file"]],
        on="event_number", how="left", suffixes=("", "_planar"),
    )
    .merge(
        spherical[["event_number", "fixed_source_speed_mps",
                   "fixed_source_coherence_score"]],
        on="event_number", how="left",
    )
)
supplement_catalogue.to_csv(TABLE_DIR / "tableS01_full_event_catalogue.csv", index=False)
geometry.to_csv(TABLE_DIR / "tableS02_receiver_geometry.csv", index=False)

manifest = {
    "main_figures": sorted(path.name for path in MAIN_FIGURE_DIR.glob("*")),
    "supplementary_figures": sorted(path.name for path in SUPPLEMENT_FIGURE_DIR.glob("*")),
    "tables": sorted(path.name for path in TABLE_DIR.glob("*")),
    "source_notebooks": list(range(1, 9)),
    "warning": "Rerun Notebooks 01–08 before freezing manuscript values.",
}
(PRODUCT_ROOT / "paper_product_manifest.json").write_text(json.dumps(manifest, indent=2) + "\n")
display(pd.Series({key: len(value) if isinstance(value, list) else value
                   for key, value in manifest.items()}, name="value").to_frame())
print("Main figures:", MAIN_FIGURE_DIR)
print("Supplementary figures:", SUPPLEMENT_FIGURE_DIR)
print("Tables:", TABLE_DIR)
